# Notebook 01 — Building the Tool Catalogue

- **Topic:** Agentic RAG — Query Understanding · 
- **Lesson:** Intent Classification → Tool Selection

So far we labelled each query with an **intent** (`kb_search`, `status_lookup`, …). 

An agent needs one more step: **which tool exactly?** 

"Why is my bill so high?" needs `get_bill` *and* `search_help_articles`, not just "status_lookup".

In this mini-series we will:
1. define the tools the Grokkers assistant can call (**this notebook**),
2. build a `gold test set` and ~1,000 `training pairs` (`query → correct tools`),
3. let an LLM pick tools, and check whether it picked the right ones,
4. train T5 to pick tools too, and use it as a local checker.

| Section | What it does |
|---|---|
| 1 | Setup |
| 2 | What a tool definition looks like |
| 3 | The 18 Grokkers tools |
| 4 | The catalogue as a table |
| 5 | How tools connect to the intent labels |
| 6 | Which tools could be confused? |
| 7 | Checking a tool call against its schema |
| 8 | Save `tools.json` |

## 1 · Setup

In [7]:
# pip install jsonschema

In [1]:
# Run once if needed:
# %pip install -q pandas scikit-learn jsonschema

import json
from pathlib import Path

import pandas as pd
from jsonschema import ValidationError, validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

TOPIC_DIR = Path.cwd().parents[2]          # tool_called → code → lesson_part_08 → topic folder
TOOLS_DIR = TOPIC_DIR / "data" / "tools"

TOOLS_DIR.mkdir(parents=True, exist_ok=True)

print("Tool data will be saved in:", TOOLS_DIR)

Tool data will be saved in: D:\gridflowai\00-ALL-COURSES-REPO\Corporate-Trainings\01-genai-developers\19-Agentic-RAG_optimizations\data\tools


## 2 · What a tool definition looks like

An LLM never *runs* a tool. It reads a list of tool **definitions** and replies with *"call this tool with these arguments"*. Our code then runs it.

Each definition has three parts:

| Part | Example | Who uses it |
|---|---|---|
| **name** | `get_bill` | Our code, to know which function to run |
| **description** | "This customer's postpaid bill for one month… Not for usage details." | **The LLM**, to decide *when* to use it |
| **parameters** | `month`: text like `2026-09` | The LLM, to fill in the arguments; our code, to check them |

> **The description is the most important part.** The LLM chooses tools almost entirely by reading descriptions. A vague description is the most common cause of a wrong tool choice. So every description below says **when to use it** and **when not to**.

Two choices that apply to every tool:
- **No phone number or customer ID parameter.** The assistant already knows who is logged in. Leaving it out means the LLM never sees or types personal data.
- **No `disconnect` tool.** Disconnection requests go to a person (`transfer_to_agent`), because the business wants a retention conversation first.

We also add three fields of our own (not sent to the LLM): which **group** the tool is in, which **intent** label it belongs to (Lesson 08), and whether it **changes the account** (those need the customer's confirmation).

In [2]:
def param(kind, description, choices=None):
    
    p = {"type": kind, "description": description}
    
    if choices is not None:
        p["enum"] = choices
        
    return p

def make_tool(name, group, intent, changes_account, description, params=None, required=None):
    
    if params is None:
        params = {}
    if required is None:
        required = list(params)            # by default every parameter is required
        
    return {
        "name": name,
        "group": group,
        "intent": intent,
        "changes_account": changes_account,
        "description": description,
        "parameters": {
            "type": "object",
            "properties": params,
            "required": required,
            "additionalProperties": False,
        },
    }

## 3 · The 18 Grokkers tools

Grouped the way a support team thinks about them: **search**, **look up** this customer's data, **act** on the account, the **catalogue**, and **conversation** tools.

### Search the help articles

In [3]:
TOOLS = []

TOOLS.append(make_tool(
    name            = "search_help_articles", 
    group           = "search", 
    intent          = "kb_search", 
    changes_account = False,
    description =
    "Search Grokkers Mobile help articles: how something works, how to do something yourself, "
    "how to fix a common problem, what a plan or feature includes. "
    "Not for this customer's own balance, bill, usage or status.",
    params =
    {"query": param("string", "What to search for, in plain words")},
))

### Look up this customer's data

In [4]:
TOOLS.append(make_tool(name=    
    "get_balance", 
    group           = "lookup",
    intent          = "status_lookup",
    changes_account = False,
    description     = "This prepaid customer's main balance in rupees and the plan's validity end date. "
                      "Not for data left in packs (use get_active_packs) or past recharges (use get_recharge_history).",
))

TOOLS.append(make_tool(
    name            = "get_active_packs",
    group           = "lookup",
    intent          = "status_lookup",
    changes_account = False,
    description     = "Plans, packs and data boosters active on this customer's number, with data, calls and SMS left "
                      "and when each expires. Use when the customer asks what they have left or why a pack ran out.",
))

TOOLS.append(make_tool(
    name            = "get_bill",
    group           = "lookup",
    intent          = "status_lookup",
    changes_account = False,
    description     = "This postpaid customer's bill for one month: total, itemised charges, taxes and due date. "
                      "Use for questions about a bill amount or bill charges. Not for day-by-day usage (use get_usage).",
    params            = {"month": param("string", "Bill month as YYYY-MM, e.g. 2026-09")},
))

TOOLS.append(make_tool(
    name            = "get_usage",
    group           = "lookup",
    intent          = "status_lookup",
    changes_account = False,
    description     = "This customer's usage records (data, calls, SMS, roaming) for a recent period. "
                      "Use when the customer asks how much they used or where their data went. Not for bill totals (use get_bill).",
    params            = {"period": param("string", "Time period", ["today", "last_7_days", "last_30_days"]),
     "usage_type": param("string", "Kind of usage", ["data", "calls", "sms", "roaming", "all"])},
))

TOOLS.append(make_tool(
    name            = "get_recharge_history",
    group           = "lookup",
    intent          = "status_lookup",
    changes_account = False,
    description     = "This prepaid customer's recent recharges and every amount deducted from the balance "
                      "(packs, subscriptions, caller tunes). Use for failed or missing recharges and unexplained deductions.",
    params            = {"last_n": param("integer", "How many recent transactions to return, 1 to 20")},
))

TOOLS.append(make_tool(
    name            = "get_kyc_status",
    group           = "lookup",
    intent          = "status_lookup",
    changes_account = False,
    description     = "Whether this customer's KYC (identity) verification is complete, pending or rejected, and why.",
))

TOOLS.append(make_tool(
    name            = "get_port_status",
    group           = "lookup",
    intent          = "status_lookup",
    changes_account = False,
    description     = "Status of this customer's number porting request, into Grokkers or out to another operator. "
                      "Not for how to port (use search_help_articles).",
    params            = {"direction": param("string", "Porting direction", ["port_in", "port_out"])},
))

TOOLS.append(make_tool(
    name            = "get_barring_reason",
    group           = "lookup",
    intent          = "status_lookup",
    changes_account = False,
    description     = "Whether outgoing or incoming services on this number are barred (blocked) and the reason, "
                      "e.g. unpaid bill, KYC pending, suspected misuse.",
))

TOOLS.append(make_tool(
    name            = "check_network_outage",
    group           = "lookup",
    intent          = "status_lookup",
    changes_account = False,
    description     = "Known network outages or maintenance at a location right now. Use when the customer reports "
                      "no signal, no service or poor network in an area, especially if it started suddenly.",
    params            = {"location": param("string", "Area, city or PIN code the customer mentioned, or 'customer_home' if none")},
))

### Act on the account (these change something, so the customer must confirm first)

In [5]:
TOOLS.append(make_tool(name="activate_pack", 
                       group="action", 
                       intent="account_action", 
                       changes_account=True,
                        description="Activate a plan, pack or data booster on this customer's number. "
                                    "Only when the customer clearly asks for it to be done. Not for 'how do I activate' questions.",
                        params={"pack_id": param("string", "Pack name or ID as the customer said it, e.g. 'Smart 599' or '10 GB booster'")},
))

TOOLS.append(make_tool(
    name="deactivate_service",
    group="action",
    intent="account_action",
    changes_account=True,
    description="Stop a service or subscription on this customer's number: caller tune, international roaming, "
                "a value-added service, or a pack's auto-renewal. Not for disconnecting the whole number (use transfer_to_agent).",
    params={"service": param("string", "The service to stop, in plain words")},
))

TOOLS.append(make_tool(
    name="change_bill_email",
    group="action",
    intent="account_action",
    changes_account=True,
    description="Change the email address this customer's postpaid bill is sent to.",
    params={"email": param("string", "The new email address")},
))

TOOLS.append(make_tool(
    name="block_sim",
    group="action",
    intent="account_action",
    changes_account=True,
    description="Block this customer's SIM immediately because the phone or SIM is lost or stolen. "
                "Not for a SIM blocked by a wrong PIN (use search_help_articles for PUK steps).",
    params={"reason": param("string", "Why the SIM must be blocked", ["lost", "stolen"])},
))

TOOLS.append(make_tool(
    name="raise_complaint",
    group="action",
    intent="account_action",
    changes_account=True,
    description="Log a formal complaint with a ticket number, for a problem that needs follow-up by the Grokkers team.",
    params={"category": param("string", "Complaint area", ["network", "billing", "recharge", "service", "other"]),
            "details": param("string", "Short summary of the problem")},
))

### Search the plan catalogue

In [6]:
TOOLS.append(make_tool(
    name="search_plan_catalogue",
    group="catalogue",
    intent="structured_query",
    changes_account=False,
    description="Filter, compare or list plans and packs across the Grokkers catalogue by price, data, validity, 5G "
                "or country. Not for a question about one named plan (use search_help_articles).",
    params={"plan_type": param("string", "Kind of product", ["prepaid", "postpaid", "roaming_pack", "data_booster", "any"]),
            "max_price": param("integer", "Highest price in rupees, if the customer gave one"),
            "needs_5g": param("boolean", "True if the customer wants 5G"),
     "country": param("string", "Destination country, for roaming packs")},
    required=["plan_type"],
))

### Conversation tools

In [7]:
TOOLS.append(make_tool(
    name="transfer_to_agent",
    group="conversation",
    intent="human_handoff",
    changes_account=False,
    description="Hand the conversation to a human agent. Use for threats or safety issues, requests to disconnect or "
                "close the number, problems the customer says are repeated, or when the customer asks for a person.",
    params={"reason": param("string", "Why a person is needed",
                            ["threat_or_safety", "disconnection", "repeat_complaint", "customer_request", "other"]),
            "summary": param("string", "One-line summary for the agent")},
))

TOOLS.append(make_tool(
    name="ask_clarifying_question",
    group="conversation",
    intent="clarify",
    changes_account=False,
    description="Ask the customer one short question when the message has two or more possible meanings that "
                "need different answers. Not for questions that can already be answered.",
    params={"question": param("string", "The question to ask the customer")},
))

print(len(TOOLS), "tools defined")

18 tools defined


And **no tool at all** is also a valid choice: greetings, thanks, and requests the assistant must refuse ("write my essay", "who won the match?"). The assistant simply replies. In the training data this will be written as `none`.

## 4 · The catalogue as a table

In [8]:
rows = []
for tool in TOOLS:
    rows.append({
        "tool": tool["name"],
        "group": tool["group"],
        "intent": tool["intent"],
        "changes account": tool["changes_account"],
        "parameters": ", ".join(tool["parameters"]["properties"]) or "-",
    })

catalogue = pd.DataFrame(rows)
catalogue

,tool,group,intent,changes account,parameters
0,search_help_articles,search,kb_search,False,query
1,get_balance,lookup,status_lookup,False,-
2,get_active_packs,lookup,status_lookup,False,-
3,get_bill,lookup,status_lookup,False,month
4,get_usage,lookup,status_lookup,False,"period, usage_type"
5,get_recharge_history,lookup,status_lookup,False,last_n
6,get_kyc_status,lookup,status_lookup,False,-
7,get_port_status,lookup,status_lookup,False,direction
8,get_barring_reason,lookup,status_lookup,False,-
9,check_network_outage,lookup,status_lookup,False,location


In [9]:
catalogue["group"].value_counts().to_frame("tools")

,tools
group,
lookup,9
action,5
conversation,2
search,1
catalogue,1


## 5 · How tools connect to the intent labels

Each intent label from Lesson 08 now opens up into one or more tools. The intent says *what kind of help*; the tool says *exactly which system to call*.

In [10]:
intent_rows = []
for intent in catalogue["intent"].unique():
    tools_for_intent = catalogue[catalogue["intent"] == intent]["tool"].tolist()
    intent_rows.append({"intent (Lesson 08)": intent,
                        "number of tools": len(tools_for_intent),
                        "tools": ", ".join(tools_for_intent)})

intent_rows.append({"intent (Lesson 08)": "refuse / small_talk", "number of tools": 0, "tools": "none (reply directly)"})
pd.DataFrame(intent_rows)

,intent (Lesson 08),number of tools,tools
0,kb_search,1,search_help_articles
1,status_lookup,9,"get_balance, get_active_packs, get_bill, get_usage, get_recharge_history, get_kyc_status, get_port_status, get_barri..."
2,account_action,5,"activate_pack, deactivate_service, change_bill_email, block_sim, raise_complaint"
3,structured_query,1,search_plan_catalogue
4,human_handoff,1,transfer_to_agent
5,clarify,1,ask_clarifying_question
6,refuse / small_talk,0,none (reply directly)


`status_lookup` alone opens into **9 tools**. That's where most wrong choices will happen: the intent can be right while the tool is wrong.

## 6 · Which tools could be confused?

If two descriptions use similar words, an LLM (or T5) can mix the tools up. We can measure this before any model runs: turn each description into a TF-IDF vector and compare every pair with **cosine similarity** (0 = no shared words, 1 = identical).

In [11]:
names = []
descriptions = []
for tool in TOOLS:
    names.append(tool["name"])
    descriptions.append(tool["description"])

vectors = TfidfVectorizer(stop_words="english").fit_transform(descriptions)
similarity = cosine_similarity(vectors)

pairs = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        pairs.append({"tool A": names[i], "tool B": names[j], "similarity": round(similarity[i, j], 2)})

pairs = pd.DataFrame(pairs).sort_values("similarity", ascending=False)
pairs.head(10)

,tool A,tool B,similarity
34,get_active_packs,get_usage,0.35
20,get_balance,get_recharge_history,0.30
17,get_balance,get_active_packs,0.27
40,get_active_packs,activate_pack,0.25
30,get_balance,search_plan_catalogue,0.20
88,get_kyc_status,get_barring_reason,0.20
45,get_active_packs,search_plan_catalogue,0.18
67,get_usage,activate_pack,0.17
19,get_balance,get_usage,0.15
6,search_help_articles,get_port_status,0.15


**Read this list with care.** TF-IDF only sees **shared words**, not meaning, so it makes two kinds of mistakes:

| Mistake | Example from the table |
|---|---|
| Flags pairs that share words but are easy to tell apart | `get_usage` ↔ `activate_pack` both mention "data", but one looks up and one acts |
| Misses pairs that mean similar things in different words | `get_bill` ↔ `get_usage` ("why is my bill high?" vs "where did my data go?") |

So the final list is **the automatic top pairs, reviewed by a person**: drop the ones that aren't real risks, add the ones the words missed. That's normal practice: the measurement suggests, a person decides.

In [12]:
top_pairs = []
for i, row in pairs.head(8).iterrows():
    top_pairs.append([row["tool A"], row["tool B"]])

# reviewer: shared words, but easy to tell apart
NOT_A_RISK = [
    ["get_balance", "search_plan_catalogue"],
    ["get_active_packs", "activate_pack"],
    ["get_usage", "activate_pack"],
    ["get_active_packs", "search_plan_catalogue"],
]

# reviewer: similar meaning, different words
MEANING_PAIRS = [
    ["get_bill", "get_usage"],
    ["search_help_articles", "search_plan_catalogue"],
    ["raise_complaint", "transfer_to_agent"],
    ["get_recharge_history", "get_bill"],
]

CONFUSABLE = []
for pair in top_pairs:
    if pair not in NOT_A_RISK:
        CONFUSABLE.append(pair)
for pair in MEANING_PAIRS:
    CONFUSABLE.append(pair)

for a, b in CONFUSABLE:
    print(f"{a:24} <-> {b}")

get_active_packs         <-> get_usage
get_balance              <-> get_recharge_history
get_balance              <-> get_active_packs
get_kyc_status           <-> get_barring_reason
get_bill                 <-> get_usage
search_help_articles     <-> search_plan_catalogue
raise_complaint          <-> transfer_to_agent
get_recharge_history     <-> get_bill


These pairs will get **near-miss examples** in the training data and extra test queries. When we evaluate the LLM later, its mistakes should mostly land here. If they do, the first fix is a clearer **description**, not a different model.

## 7 · Checking a tool call against its schema

When the LLM answers "call `get_bill` with `month = 2026-09`", the first and cheapest check is: **does that tool exist, and are the arguments valid?** The `jsonschema` package checks arguments against the `parameters` we wrote. This is the first layer of the runtime verifier we'll build later.

In [13]:
TOOLS_BY_NAME = {}
for tool in TOOLS:
    TOOLS_BY_NAME[tool["name"]] = tool


def check_call(name, arguments):
    if name not in TOOLS_BY_NAME:
        return "unknown tool"
    try:
        validate(instance=arguments, schema=TOOLS_BY_NAME[name]["parameters"])
    except ValidationError as error:
        return "bad arguments: " + error.message
    return "ok"


examples = [
    ("get_bill", {"month": "2026-09"}),
    ("get_bill", {}),                                          # month missing
    ("get_usage", {"period": "last_year", "usage_type": "data"}),  # not an allowed period
    ("block_sim", {"reason": "lost"}),
    ("cancel_my_number", {}),                                  # tool doesn't exist
]

for name, arguments in examples:
    print(f"{name:20} {str(arguments):45} → {check_call(name, arguments)}")

get_bill             {'month': '2026-09'}                          → ok
get_bill             {}                                            → bad arguments: 'month' is a required property
get_usage            {'period': 'last_year', 'usage_type': 'data'} → bad arguments: 'last_year' is not one of ['today', 'last_7_days', 'last_30_days']
block_sim            {'reason': 'lost'}                            → ok
cancel_my_number     {}                                            → unknown tool


A schema check catches made-up tools and malformed arguments. It **can't** tell whether `get_bill` was the right tool for the question, or whether September was the right month. That needs a gold set (Notebook 02) and a second opinion (T5, later).

## 8 · Save `tools.json`

We save two things:
- **The full catalogue**, with our own fields (group, intent, changes_account), for our code.
- **The OpenAI version**, only `name`, `description` and `parameters`, in the format the Responses API expects. The LLM never sees our own fields.

In [14]:
openai_tools = []
for tool in TOOLS:
    openai_tools.append({
        "type": "function",
        "name": tool["name"],
        "description": tool["description"],
        "parameters": tool["parameters"],
    })

TOOL_ORDER = []                      # fixed order, used when writing several tools: "get_bill, search_help_articles"
for tool in TOOLS:
    TOOL_ORDER.append(tool["name"])

catalogue_file = {
    "tools": TOOLS,
    "tool_order": TOOL_ORDER,
    "confusable_pairs": CONFUSABLE,
    "no_tool_label": "none",
}

with open(TOOLS_DIR / "tools.json", "w", encoding="utf-8") as f:
    json.dump(catalogue_file, f, indent=2, ensure_ascii=False)

with open(TOOLS_DIR / "tools_openai.json", "w", encoding="utf-8") as f:
    json.dump(openai_tools, f, indent=2, ensure_ascii=False)

print("Saved tools.json and tools_openai.json in", TOOLS_DIR)
print("Tool order:", TOOL_ORDER)

Saved tools.json and tools_openai.json in D:\gridflowai\00-ALL-COURSES-REPO\Corporate-Trainings\01-genai-developers\19-Agentic-RAG_optimizations\data\tools
Tool order: ['search_help_articles', 'get_balance', 'get_active_packs', 'get_bill', 'get_usage', 'get_recharge_history', 'get_kyc_status', 'get_port_status', 'get_barring_reason', 'check_network_outage', 'activate_pack', 'deactivate_service', 'change_bill_email', 'block_sim', 'raise_complaint', 'search_plan_catalogue', 'transfer_to_agent', 'ask_clarifying_question']


## What's next

**Notebook 02 · Gold test set:** about 100 hand-written customer messages, each with its **correct** tools (and accepted alternatives where more than one answer is right). It includes the 81 intent test queries from Lesson 08, now labelled with tools. It's written before any model runs, so neither the LLM nor T5 can influence it.